In [3]:
# bin_to_wav.py
import wave
import numpy as np
from pathlib import Path

# ========= 여길 딱! 한 번만 바꾸면 됩니다 =========
BIN_DIR    = Path("/home/jonghyuk/Desktop/micloaker/bin")  # .bin들이 있는 폴더
WAV_DIR    = Path("/home/jonghyuk/Desktop/micloaker/wav")  # .wav들이 저장될 폴더

SAMPLE_RATE= 8000    # 녹음 샘플레이트(Hz)
CHANNELS   = 1       # 채널 수 (인터리브 가정)
DTYPE      = "<f8"   # BIN 데이터형: float64 little-endian (ULDAQ 기본 더블)
REMOVE_DC  = True    # 채널별 DC 오프셋 제거
SCALE_MODE = "range"  # 'peak' 또는 'range'
FULL_SCALE_VOLTS = 10.0  # SCALE_MODE='range'일 때 ±전압을 ±1로 매핑
RECURSIVE  = False   # 하위 폴더까지 처리하려면 True

OVERWRITE  = False    # 기존 wav가 있으면 덮어쓸지 여부
# ===============================================

def load_bin(bin_path: Path, dtype: str, ch: int) -> np.ndarray:
    if not bin_path.exists():
        raise FileNotFoundError(f"BIN 파일이 없습니다: {bin_path}")
    raw = np.fromfile(bin_path, dtype=dtype)
    if raw.size % ch != 0:
        raise ValueError(
            f"채널 수({ch})로 나눈 길이가 정수가 아닙니다. samples={raw.size}, ch={ch}, dtype={dtype}"
        )
    return raw.reshape(-1, ch) if ch > 1 else raw.reshape(-1, 1)

def to_pcm16(volts: np.ndarray, mode="peak", full_scale_volts=10.0) -> np.ndarray:
    x = volts.astype(np.float64, copy=False)
    if mode == "range":
        scale = 1.0 / float(full_scale_volts) if full_scale_volts > 0 else 1.0
        x = np.clip(x * scale, -1.0, 1.0)
    else:
        peak = float(np.max(np.abs(x))) if x.size else 1.0
        scale = (0.999 / peak) if peak > 0 else 1.0
        x = np.clip(x * scale, -1.0, 1.0)
    return np.int16(np.round(x * 32767.0))

def write_wav(path: Path, pcm16: np.ndarray, sr: int, ch: int):
    pcm16 = np.ascontiguousarray(pcm16)  # 인터리브 바이트 연속성 보장
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(ch)
        wf.setsampwidth(2)  # 16-bit
        wf.setframerate(sr)
        wf.writeframes(pcm16.tobytes())

def process_one(bin_path: Path):
    # BIN_DIR 기준 상대 경로를 유지해서 WAV_DIR에 저장
    rel_path = bin_path.relative_to(BIN_DIR)
    wav_path = (WAV_DIR / rel_path).with_suffix(".wav")

    # wav 저장 폴더가 없으면 자동 생성
    wav_path.parent.mkdir(parents=True, exist_ok=True)

    data = load_bin(bin_path, dtype=DTYPE, ch=CHANNELS)

    if REMOVE_DC:
        data = data - data.mean(axis=0, keepdims=True)

    pcm16 = to_pcm16(data, mode=SCALE_MODE, full_scale_volts=FULL_SCALE_VOLTS)
    write_wav(wav_path, pcm16, SAMPLE_RATE, CHANNELS)

    n_samples = pcm16.shape[0]
    dur = n_samples / float(SAMPLE_RATE)
    peak_v = float(np.max(np.abs(data))) if data.size else 0.0

    info = {
        "bin": bin_path,
        "wav": wav_path,
        "duration_s": dur,
        "sample_rate": SAMPLE_RATE,
        "channels": CHANNELS,
        "peak_v": peak_v,
        "n_samples": n_samples,
    }

    return info

def main():
    bins = sorted(
        (BIN_DIR.rglob("*.bin") if RECURSIVE else BIN_DIR.glob("*.bin")),
        key=lambda p: str(p)
    )

    if not bins:
        print(f"[INFO] 대상 .bin 파일이 없습니다: {BIN_DIR} (RECURSIVE={RECURSIVE})")
        return

    existing = []
    todo = []

    for bin_path in bins:
        rel_path = bin_path.relative_to(BIN_DIR)
        wav_path = (WAV_DIR / rel_path).with_suffix(".wav")

        if wav_path.exists() and not OVERWRITE:
            existing.append((bin_path, wav_path))
        else:
            todo.append((bin_path, wav_path))

    total = len(bins)
    n_existing = len(existing)
    n_todo = len(todo)

    print("=" * 90)
    print("[SCAN]")
    print(f"BIN 폴더              : {BIN_DIR}")
    print(f"WAV 폴더              : {WAV_DIR}")
    print(f"전체 bin 파일 수       : {total}")
    print(f"이미 wav 존재          : {n_existing}")
    print(f"이번에 변환할 파일 수  : {n_todo}")
    print(f"OVERWRITE             : {OVERWRITE}")
    print("=" * 90)

    if n_todo == 0:
        print("[DONE] 새로 변환할 파일이 없습니다.")
        return

    processed_infos = []
    failed = []

    for i, (bin_path, wav_path) in enumerate(todo, 1):
        try:
            print(f"[{i}/{n_todo}] 변환 중: {bin_path.name} → {wav_path.name}")
            info = process_one(bin_path)
            processed_infos.append(info)
        except Exception as e:
            print(f"[ERROR] {bin_path.name} → {e}")
            failed.append((bin_path, e))

    print()
    print("=" * 90)
    print("[PROCESSED FILES]")
    print(f"{'#':>3}  {'BIN':<30} {'WAV':<30} {'dur(s)':>8} {'peak(V)':>10} {'samples':>10}")
    print("-" * 90)

    for idx, info in enumerate(processed_infos, 1):
        print(
            f"{idx:>3}  "
            f"{info['bin'].name:<30} "
            f"{info['wav'].name:<30} "
            f"{info['duration_s']:>8.2f} "
            f"{info['peak_v']:>10.6f} "
            f"{info['n_samples']:>10}"
        )

    print("=" * 90)
    print("[SUMMARY]")
    print(f"전체 bin 파일 수       : {total}")
    print(f"이미 wav 존재          : {n_existing}")
    print(f"새로 변환 성공         : {len(processed_infos)}")
    print(f"변환 실패              : {len(failed)}")

    if failed:
        print()
        print("[FAILED FILES]")
        for bin_path, e in failed:
            print(f"- {bin_path.name}: {e}")

if __name__ == "__main__":
    main()


[SCAN]
BIN 폴더              : /home/jonghyuk/Desktop/micloaker/bin
WAV 폴더              : /home/jonghyuk/Desktop/micloaker/wav
전체 bin 파일 수       : 8
이미 wav 존재          : 8
이번에 변환할 파일 수  : 0
OVERWRITE             : False
[DONE] 새로 변환할 파일이 없습니다.
